#### Контекст к табличке
Срочные депозиты являются основным источником дохода для банка. Срочный депозит (вклад) - это вложение денежных средств в финансовое учреждение. Ваши деньги инвестируются по согласованной процентной ставке в течение определенного периода времени. У банка есть различные планы по продаже срочных депозитов своим клиентам, такие как маркетинг по электронной почте, реклама, телефонный маркетинг и цифровой маркетинг.

Телефонные маркетинговые кампании по-прежнему остаются одним из наиболее эффективных способов привлечения внимания к людям. Однако они требуют огромных инвестиций, поскольку для их проведения нанимаются крупные колл-центры. Следовательно, крайне важно заранее определить клиентов, которые с наибольшей вероятностью перейдут к вам, чтобы их можно было целенаправленно привлечь с помощью звонков.

Данные относятся к кампаниям прямого маркетинга (телефонным звонкам) португальского банковского учреждения. Цель классификации - предсказать, подпишется ли клиент на срочный депозит (переменная y).

## **шаг 1** | *Информация по таблице*
* Строк: 45211
* Столбцов: 17
* Типы данных: int(64) - 7, str - 10
* Дубликатов: 0

In [4]:
import pandas as pd

data = pd.read_csv(filepath_or_buffer='bank.csv', sep=';')

data.info()

data.shape

data.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


np.int64(0)

## **шаг 2** | *Замена дублей, замена пустых ячеек*
* Было удалено 0 дублей, значения **None** заменены на **unknown**
* Замена всех значений **no** на **False** и **yes** на **True**

Дополнительно был проведен анализ столбца **default**, мне казалось, что все значеняи там - no, но оказалось:

no     44396
yes      815

In [5]:
data.drop_duplicates(keep='last')

data.fillna(value='unknown')

data = data.rename(columns={'marital': 'family', 'y': 'get_deposit'})

data.replace(to_replace=['yes', 'no'], value=[True, False])

,age,job,family,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,get_deposit
0,58,management,married,tertiary,False,2143,True,False,unknown,5,may,261,1,-1,0,unknown,False
1,44,technician,single,secondary,False,29,True,False,unknown,5,may,151,1,-1,0,unknown,False
2,33,entrepreneur,married,secondary,False,2,True,True,unknown,5,may,76,1,-1,0,unknown,False
3,47,blue-collar,married,unknown,False,1506,True,False,unknown,5,may,92,1,-1,0,unknown,False
4,33,unknown,single,unknown,False,1,False,False,unknown,5,may,198,1,-1,0,unknown,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,False,825,False,False,cellular,17,nov,977,3,-1,0,unknown,True
45207,71,retired,divorced,primary,False,1729,False,False,cellular,17,nov,456,2,-1,0,unknown,True
45208,72,retired,married,secondary,False,5715,False,False,cellular,17,nov,1127,5,184,3,success,True
45209,57,blue-collar,married,secondary,False,668,False,False,telephone,17,nov,508,4,-1,0,unknown,False


## **шаг 3** | Анализ предоставленных данных
Здесь мною будут выдвинуты гипотезы, которые я либо докажу, либо опровергну.

#### Гипотеза 1
гласит: Больше всего денег (среднее значение за 12 месяцев) на счете у тех, кто имиеет высшее учебное образование.

ответ: 
Отбросив строку с *unknown* образованием мы получим, что, во первых, людей с средним школьным образованием (11 классов получается) - больше, чем людей с высшем образованием. 
По профессиям видно, что люди с высшим образованием идут в менеджмент и управление, а все остальные, скорее всего, работают в офисе, как раз под их руководством.
Что касается заработка - несмотря на то, что людей с одним школьным образованием больше, чем всех остальных вместе взятых, у группы с высшим образованием на счете хранится больше денрежных средств, чем у остальных. Возможными причинами могут быть более высокий уровень дохода или особенности финансового поведения. Для подтверждения необходимы дополнительные данные.

In [6]:
# гипотеза 1
data.head()

money_value = data.groupby(by=['education'])[['age', 'job', 'family', 'balance']].agg(
    age_mean=('age', 'mean'),
    balance_median=('balance', 'median'),
    total_entries=('age', 'size'),
    popular_job=('job', lambda x: x.mode()[0]),
    popular_family=('family', lambda x: x.mode()[0])
)

money_value

,age_mean,balance_median,total_entries,popular_job,popular_family
education,,,,,
primary,45.865567,403.0,6851,blue-collar,married
secondary,39.964270,392.0,23202,blue-collar,married
tertiary,39.593640,577.0,13301,management,married
unknown,44.510501,568.0,1857,blue-collar,married


#### Гипотеза 2
гласит: Люди, кто не имеет кредитов охотнее соглашаются на вклад.
ответ: 
По полученным данным видно, что люди не имеющие ипотеки берут вклад на 50% активнее, это может быть обусловленно тем, 
что нет высокоприоритетного и тяжелого по затратам обязательства перед банком - соответсвенно нет свободных средств. 
Однако, людей что имеют обычный кредит и берут депозит на 5% меньше, чем людей, что имеют и ипотеку, и кредит. 
Склоняюсь к тому, что это не показательные данные.
По таблице можно сделать достаточно много выводов. Основной сделан.

In [7]:
deposit_agree = data.groupby(by=['housing', 'loan', 'get_deposit']).agg({
    'get_deposit': 'size'
})

deposit_agree.head(10)

get_deposit
housing loan get_deposit             
no      no   no                 14069
             yes                 3135
        yes  no                  2658
             yes                  219
yes     no   no                 19093
             yes                 1670
        yes  no                  4102
             yes                  265

## Аномалии
Протсейшие и интересные наблюдения.
- Наблюдение 1: половина пользователей датасета имеет баланс меньше медианного показателя. Это может говорить о том, что аккаунт просто заброшен. Таким пользователям нужны выгодные предложения для карт, а не для вклада. 
- Наблюдение 2: Женатые имеют на 37% больше денег на счетах, чем разведенные.
- Наблюдение 3: Пользователей в возрасте 20 лет и младше почти столько же, сколько тех, кому 21 год.

In [ ]:
# Наблюдение 1
data.loc[data['balance'] < data['balance'].median()] 

# Наблюдение 2
data.loc[(data['family'] == 'married') | (data['family'] == 'divorced')].groupby(by=['family']).agg({
    'balance': 'median'
}).head() 

# Наблюдение 3
data.loc[data['age'] <= 20].groupby(by=['age']).size()
data.loc[data['age'] == 21].groupby(by=['age']).size()

age
21    79
dtype: int64

## Вывод
В рамках проекта был проведен анализ данных маркетинговой компании банка по привлечению пользователей к оформлению вкладов.

Первичная обработка и анализ показали отсутсвие дубликатов, что говорит об отсутствии ошибок внутри банковской системы. Существенных проблем с данными обнаружено не было.

В ходе анализа были проверены несколько гипотез. Выявлено, что клиенты с высшим образованием имеют более высокий медианный баланс по сравнению с другими группами клиентов. Также было замечено, что наличие кредитных обязательств может влиять на склонность клиента к открытию депозита.

Полученные результаты позволяют предположить, что социально-демографические характеристики и финансовое положение клиента связаны с его инвестиционным поведением. Однако для подтверждения причинно-следственных связей требуется дополнительный анализ и более подробные данные о доходах клиентов.